# Passo 3 – Pré-processamento e partição dos dados

Este notebook:

1. Carrega as três bases já limpas (Fake.Br, FakeRecogna e o conjunto eleitoral FactPolCheckBr).
2. Normaliza e tokeniza os textos.
3. Remove stopwords, **mantendo as palavras de negação** (não, nem, nunca, jamais, nenhum, sem).
4. Gera duas versões do texto: **sem stemming** e **com stemming (RSLP)**. As duas serão comparadas no Passo 4.
5. Divide o Fake.Br em treino e teste **sem separar os pares** (notícia falsa e verdadeira do mesmo par ficam do mesmo lado).
6. Mostra estatísticas para a seção de metodologia.

**Por que duas versões?** O RSLP reduz *eleições* a *ele*, *eleitoral* a *eleitor* e *eleitores* a *elei*. Em um estudo sobre eleições, isso pode misturar palavras diferentes. Por isso testamos com e sem stemming.

**Pré-requisito:** notebooks 00, 01, 02 e 02b executados.

In [ ]:
import os, re, unicodedata
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_TCC = '/content/drive/MyDrive/TCC'
except ImportError:
    PASTA_TCC = os.path.abspath('TCC')

PROC = os.path.join(PASTA_TCC, 'dados', 'processados')
RES = os.path.join(PASTA_TCC, 'resultados')
SEED = 42

fake = pd.read_csv(os.path.join(PROC, 'fakebr_limpo.csv'))
reco = pd.read_csv(os.path.join(PROC, 'recogna_limpo.csv'))
fpc = pd.read_csv(os.path.join(PROC, 'fpc_eleitoral_final.csv'))
for nome, d in [('Fake.Br', fake), ('FakeRecogna', reco), ('FactPolCheckBr (eleitoral)', fpc)]:
    print(f'{nome}: {len(d)} textos | rótulos: {d.rotulo.value_counts().to_dict()}')

## 1. Funções de pré-processamento

Regras:

- Minúsculas.
- URLs, menções (@) e hashtags viram marcadores (`urltoken`, `mencao`, `hashtag`) ou saem do texto. Aqui eles **saem**, para não criar atalhos.
- Números viram o marcador `num`.
- Pontuação sai. Os acentos **ficam** (a lista de stopwords e o RSLP usam acento).
- Stopwords saem, **menos as negações**.

Os trechos `(sic)`, `[sic]` e `(...)` são marcas de quem escreveu a checagem. Eles saem.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

NEGACOES = {'não', 'nem', 'nunca', 'jamais', 'nenhum', 'nenhuma', 'sem', 'nada', 'ninguém'}
STOP = set(stopwords.words('portuguese')) - NEGACOES
STEM = RSLPStemmer()

RE_URL = re.compile(r'(https?://\S+|www\.\S+)', re.I)
RE_MENCAO = re.compile(r'@\w+')
RE_HASHTAG = re.compile(r'#\w+')
RE_MARCA = re.compile(r'\(sic\.?\)|\[sic\.?\]|\(\.\.\.\)|\[\.\.\.\]', re.I)
RE_NUM = re.compile(r'\d+([.,]\d+)*')
RE_NAOLETRA = re.compile(r'[^a-zA-ZÀ-ÿ ]')

def normalizar(texto):
    t = str(texto)
    t = unicodedata.normalize('NFC', t)
    t = RE_URL.sub(' ', t)
    t = RE_MARCA.sub(' ', t)
    t = RE_MENCAO.sub(' ', t)
    t = RE_HASHTAG.sub(' ', t)
    t = t.lower()
    t = RE_NUM.sub(' num ', t)
    t = RE_NAOLETRA.sub(' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def tokenizar(texto):
    return [p for p in normalizar(texto).split() if len(p) > 1 or p == 'e']

def sem_stopwords(tokens):
    return [t for t in tokens if t not in STOP]

def preparar(texto):
    toks = sem_stopwords(tokenizar(texto))
    return ' '.join(toks), ' '.join(STEM.stem(t) for t in toks)

# Teste rápido
ex = 'O TSE disse que NÃO houve fraude em 2022 (sic) https://x.com/a #urna @fulano. Eleitores votaram!'
print(normalizar(ex)); print(preparar(ex))

In [ ]:
# Conferência do stemming em palavras do tema
for p in ['eleições', 'eleição', 'eleitoral', 'eleitores', 'urnas', 'urna', 'fraude', 'votos', 'votar', 'presidente', 'presidência']:
    print(f'{p:12} -> {STEM.stem(p)}')

## 2. Aplicação nas três bases

In [ ]:
for d in (fake, reco, fpc):
    saida = d['texto'].apply(preparar)
    d['texto_proc'] = [s[0] for s in saida]
    d['texto_stem'] = [s[1] for s in saida]
    d['n_palavras_orig'] = d['texto'].str.split().str.len()
    d['n_tokens_proc'] = d['texto_proc'].str.split().str.len()

vazios = {n: int((d['n_tokens_proc'] == 0).sum()) for n, d in [('fake', fake), ('reco', reco), ('fpc', fpc)]}
print('Textos que ficaram vazios após o processamento:', vazios)
assert sum(vazios.values()) == 0, 'Há textos vazios; investigar antes de continuar'

## 3. Partição do Fake.Br (treino e teste interno)

O Fake.Br tem **pares**: cada notícia falsa tem uma verdadeira sobre o mesmo assunto. Se os dois lados do par ficarem um no treino e outro no teste, o modelo vê o assunto antes e o resultado fica otimista. Por isso a divisão é **por par** (`par_id`).

- 80% dos pares para treino, 20% para teste.
- A validação cruzada do Passo 4 também vai usar `par_id` como grupo (`StratifiedGroupKFold`).

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
idx_tr, idx_te = next(gss.split(fake, fake['rotulo'], groups=fake['par_id']))
fake['particao'] = 'treino'
fake.loc[fake.index[idx_te], 'particao'] = 'teste'

# Verificações
pares_tr = set(fake.loc[fake.particao == 'treino', 'par_id'])
pares_te = set(fake.loc[fake.particao == 'teste', 'par_id'])
assert pares_tr.isdisjoint(pares_te), 'Vazamento: par nos dois lados'
print(pd.crosstab(fake['particao'], fake['rotulo'], margins=True))
print('Pares no treino:', len(pares_tr), '| pares no teste:', len(pares_te))

# FakeRecogna: não tem pares, divisão estratificada simples (usada só no experimento de viés)
reco['particao'] = 'treino'
_, i_te = train_test_split(reco.index, test_size=0.20, stratify=reco['rotulo'], random_state=SEED)
reco.loc[i_te, 'particao'] = 'teste'
print(pd.crosstab(reco['particao'], reco['rotulo'], margins=True))

## 4. Estatísticas para a metodologia

Atenção ao **tamanho do texto**. Se falsos e verdadeiros têm tamanhos muito diferentes, o modelo pode aprender o tamanho e não o conteúdo. O conjunto eleitoral tem textos curtos (mensagens), e o Fake.Br tem notícias longas.

In [ ]:
def resumo(nome, d):
    g = d.groupby('rotulo')['n_palavras_orig']
    vocab = len(set(' '.join(d['texto_proc']).split()))
    vocab_s = len(set(' '.join(d['texto_stem']).split()))
    return {'base': nome, 'textos': len(d), 'vocab_sem_stem': vocab, 'vocab_com_stem': vocab_s,
            'mediana_palavras_falso': g.median().get(1), 'mediana_palavras_verdadeiro': g.median().get(0),
            'media_palavras': round(d['n_palavras_orig'].mean(), 1)}

estat = pd.DataFrame([resumo('Fake.Br', fake), resumo('FakeRecogna', reco), resumo('FactPolCheckBr (eleitoral)', fpc)])
estat

In [ ]:
# Teste de tamanho: quanto o tamanho sozinho separa falso de verdadeiro? (AUC do número de palavras)
from sklearn.metrics import roc_auc_score
for nome, d in [('Fake.Br', fake), ('FakeRecogna', reco)]:
    auc = roc_auc_score(d['rotulo'], d['n_palavras_orig'])
    print(f'{nome}: AUC só com o tamanho do texto = {auc:.3f}  (0,5 = não separa; longe de 0,5 = há viés de tamanho)')

## 5. Salvando

In [ ]:
cols = ['base', 'texto', 'texto_proc', 'texto_stem', 'rotulo', 'n_palavras_orig', 'n_tokens_proc']
fake[cols + ['par_id', 'particao']].to_csv(os.path.join(PROC, 'fakebr_prep.csv'), index=False)
reco[cols + ['particao']].to_csv(os.path.join(PROC, 'recogna_prep.csv'), index=False)
fpc[cols + ['id', 'agencia']].to_csv(os.path.join(PROC, 'fpc_prep.csv'), index=False)
estat.to_csv(os.path.join(RES, 'quadro2_estatisticas_bases.csv'), index=False)

with open(os.path.join(RES, 'preprocessamento.txt'), 'w') as f:
    f.write(f'seed={SEED}\nstopwords_removidas={len(STOP)}\nnegacoes_mantidas={sorted(NEGACOES)}\n')
    f.write('stemmer=RSLP (nltk)\nparticao_fakebr=GroupShuffleSplit 80/20 por par_id\n')
print('Arquivos salvos em', PROC)